# University Policy Copilot — Colab launcher
Runs the Streamlit UI in Colab using the real agent.

**Before you start** (Runtime > Run all works after step 1):
1. Upload this whole folder to Colab (or its zip) — see cell 1.
2. Have ready: `vm05-ca.crt`, `vm02-ca.crt`, and the two shared API keys.

The policy corpus and gold set are already inside `data/`.

In [ ]:
# 1) Locate / unpack the project and install dependencies (~2-3 min)
import os, sys, zipfile, glob, subprocess
from pathlib import Path

if not Path("app.py").exists():
    here = next(iter(glob.glob("/content/policy-copilot-colab")), None)
    if here is None:
        z = next(iter(glob.glob("/content/policy-copilot-colab*.zip") +
                      glob.glob("/content/drive/MyDrive/**/policy-copilot-colab*.zip", recursive=True)), None)
        assert z, "Upload policy-copilot-colab.zip to /content (left file panel) and re-run."
        zipfile.ZipFile(z).extractall("/content")
    os.chdir("/content/policy-copilot-colab")
print("Working dir:", os.getcwd())
print(sorted(os.listdir("data")))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Dependencies installed.")

In [ ]:
# 2) Certificate(s) -> certs/   (one .crt is enough, e.g. vm05-ca.crt)
from google.colab import files
os.makedirs("certs", exist_ok=True)
if not list(Path("certs").glob("*.crt")):
    print("Select your certificate file (.crt):")
    for name, data in files.upload().items():
        Path("certs", Path(name).name).write_bytes(data)
CERTS = sorted(Path("certs").glob("*.crt"))
assert CERTS, "No .crt file found in certs/ - re-run this cell and choose your certificate."
print("certs/:", [c.name for c in CERTS])

In [ ]:
# 3) Keys and configuration (keys are typed here, never written to disk)
from getpass import getpass
os.environ.update({
    "POLICY_AGENT_MODE": "live",
    "POLICY_ZIP_PATH": "data/University_Policy_Document_Corpus.zip",
    # Use vm05 for the LLM if present; the embedding server uses vm02 if you have it,
    # otherwise the same certificate (as the original notebook does).
    "LLM_CA_CERT": str(next((c for c in CERTS if "vm05" in c.name), CERTS[0])),
    "EMBEDDING_CA_CERT": str(next((c for c in CERTS if "vm02" in c.name),
                                  next((c for c in CERTS if "vm05" in c.name), CERTS[0]))),
    "M4_CHECKPOINT_DB_PATH": "/content/policy_agent_checkpoints.sqlite",
})
os.environ["OPENAI_API_KEY"]  = getpass("Group LLM API key: ").strip()
os.environ["EMBEDDINGS_KEY"] = getpass("Embedding API key: ").strip()
# Optional real ticketing. Leave unset to keep support requests DEMO ONLY.
# os.environ["UNIVERSITY_SUPPORT_API_URL"] = "https://..."
# os.environ["UNIVERSITY_SUPPORT_API_TOKEN"] = getpass("Support token: ")
print("Configured. Mode:", os.environ["POLICY_AGENT_MODE"])

In [ ]:
# 4) Start the app and open a public link (Cloudflare quick tunnel, no account/password)
import re, time, subprocess, urllib.request

subprocess.run("pkill -f 'streamlit run' ; pkill -f cloudflared", shell=True)
log = open("/content/streamlit.log", "w")
app = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py", "--server.port", "8501",
     "--server.headless", "true", "--browser.gatherUsageStats", "false",
     "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"],  # needed for the Colab proxy
    stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy())

for _ in range(60):                       # wait for the server
    try:
        urllib.request.urlopen("http://localhost:8501/_stcore/health", timeout=2); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Streamlit did not start. Run: !cat /content/streamlit.log")

if not Path("/content/cloudflared").exists():
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "/content/cloudflared")
    os.chmod("/content/cloudflared", 0o755)

tun = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://localhost:8501"],
    stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)

url = None
for _ in range(60):                       # wait for the tunnel address
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m:
        url = m.group(0); break
    time.sleep(1)
assert url, "Tunnel did not start. Run: !cat /content/tunnel.log"

import socket                             # wait until the name actually resolves
host = url.split("//")[1]
for _ in range(45):
    try:
        socket.gethostbyname(host); break
    except OSError:
        time.sleep(2)
time.sleep(15)                            # extra time for global DNS to catch up

print("\nOPEN THIS LINK:", url)
print("If your browser says 'site can't be reached', wait 1 minute and reload,")
print("or try a private window / phone data. Otherwise run the alternative-proxy cell below.")
print("First load builds the policy index (~3.4k chunks) - allow a few minutes.")
print("The link is public while this session runs; do not share it.")

### If the link does not open
Run the cell below to use Colab's built-in proxy instead, or check the logs.

In [ ]:
# Alternative: Colab's own proxy (needs the app from cell 4 to be running)
from google.colab import output
output.serve_kernel_port_as_window(8501)

In [ ]:
# Logs (use this if the app shows "could not start")
print(open("/content/streamlit.log").read()[-4000:])

In [ ]:
# Stop everything
subprocess.run("pkill -f 'streamlit run' ; pkill -f cloudflared", shell=True)